In [1]:
import os, sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))

In [2]:
from src.extraction import ExtractionContext, LayoutExtractor, TextExtractor
from src.chunker import HierarchicalChunker, EmbeddingSemanticChunker, TextTilingChunker
from src.embedding import LocalEmbedder, OllamaEmbedder, OpenAIEmbedder


/home/pritesh-jha/projects/poc-to-prod/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
pdf_path = "/home/pritesh-jha/projects/poc-to-prod/main/assets/data/1706.03762v7.pdf"

In [5]:
context = ExtractionContext(file_path=pdf_path)
LayoutExtractor().extract(context)
text_records = TextExtractor().extract(context)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 360/360 [00:00<00:00, 8661.99it/s]
[INFO] 2026-04-02 19:03:29,876 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-02 19:03:29,891 [RapidOCR] download_file.py:60: File exists and is valid: /home/pritesh-jha/projects/poc-to-prod/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-04-02 19:03:29,891 [RapidOCR] main.py:53: Using /home/pritesh-jha/projects/poc-to-prod/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-04-02 19:03:29,953 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-02 19:03:29,954 [RapidOCR] download_file.py:60: File exists and is valid: /home/pritesh-jha/projects/poc-to-prod/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-04-02 19:03:29,954 [RapidOCR] main.py:53: Using /home/pritesh-jha/projects/poc-to-prod/.venv/lib/pyth

In [6]:
# Hierarchical
docs = HierarchicalChunker().chunk_records(text_records, source=context.file_path)

In [7]:
docs[0:2]

[Document(metadata={'source': '/home/pritesh-jha/projects/poc-to-prod/main/assets/data/1706.03762v7.pdf', 'page': 1, 'type': 'text', 'bbox': {'l': 18.34, 't': 627.0, 'r': 36.34, 'b': 237.0}, 'parent_id': 0, 'parent_text': 'arXiv:1706.03762v7  [cs.CL]  2 Aug 2023'}, page_content='arXiv:1706.03762v7  [cs.CL]  2 Aug 2023'),
 Document(metadata={'source': '/home/pritesh-jha/projects/poc-to-prod/main/assets/data/1706.03762v7.pdf', 'page': 1, 'type': 'text', 'bbox': {'l': 124.313, 't': 717.8198352, 'r': 487.89454240000015, 'b': 679.662512679646}, 'parent_id': 0, 'parent_text': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.'}, page_content='Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.')]

In [8]:
# Lexical semantic — zero dependencies
docs = TextTilingChunker(depth_threshold=0.3).chunk_records(text_records, source=context.file_path)
docs[0:2]

[Document(metadata={'source': '/home/pritesh-jha/projects/poc-to-prod/main/assets/data/1706.03762v7.pdf', 'page': 1, 'type': 'text', 'bbox': {'l': 18.34, 't': 627.0, 'r': 36.34, 'b': 237.0}}, page_content='arXiv:1706.03762v7  [cs.CL]  2 Aug 2023'),
 Document(metadata={'source': '/home/pritesh-jha/projects/poc-to-prod/main/assets/data/1706.03762v7.pdf', 'page': 1, 'type': 'text', 'bbox': {'l': 124.313, 't': 717.8198352, 'r': 487.89454240000015, 'b': 679.662512679646}}, page_content='Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.')]

In [9]:
# Embedding semantic — plug in any embedder
docs = EmbeddingSemanticChunker(embedder=LocalEmbedder()).chunk_records(text_records, source=context.file_path)
docs[0:2]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2717.68it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Document(metadata={'source': '/home/pritesh-jha/projects/poc-to-prod/main/assets/data/1706.03762v7.pdf', 'page': 1, 'type': 'text', 'bbox': {'l': 18.34, 't': 627.0, 'r': 36.34, 'b': 237.0}}, page_content='arXiv:1706.03762v7  [cs.CL]  2 Aug 2023'),
 Document(metadata={'source': '/home/pritesh-jha/projects/poc-to-prod/main/assets/data/1706.03762v7.pdf', 'page': 1, 'type': 'text', 'bbox': {'l': 124.313, 't': 717.8198352, 'r': 487.89454240000015, 'b': 679.662512679646}}, page_content='Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.')]

In [10]:

docs = EmbeddingSemanticChunker(embedder=OllamaEmbedder(model="nomic-embed-text-v2-moe:latest")).chunk_records(text_records, source=context.file_path)
docs[0:2]

[Document(metadata={'source': '/home/pritesh-jha/projects/poc-to-prod/main/assets/data/1706.03762v7.pdf', 'page': 1, 'type': 'text', 'bbox': {'l': 18.34, 't': 627.0, 'r': 36.34, 'b': 237.0}}, page_content='arXiv:1706.03762v7  [cs.CL]  2 Aug 2023'),
 Document(metadata={'source': '/home/pritesh-jha/projects/poc-to-prod/main/assets/data/1706.03762v7.pdf', 'page': 1, 'type': 'text', 'bbox': {'l': 124.313, 't': 717.8198352, 'r': 487.89454240000015, 'b': 679.662512679646}}, page_content='Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.')]

In [11]:
docs = EmbeddingSemanticChunker(embedder=OpenAIEmbedder()).chunk_records(text_records, source=context.file_path)
docs[0:2]

[Document(metadata={'source': '/home/pritesh-jha/projects/poc-to-prod/main/assets/data/1706.03762v7.pdf', 'page': 1, 'type': 'text', 'bbox': {'l': 18.34, 't': 627.0, 'r': 36.34, 'b': 237.0}}, page_content='arXiv:1706.03762v7  [cs.CL]  2 Aug 2023'),
 Document(metadata={'source': '/home/pritesh-jha/projects/poc-to-prod/main/assets/data/1706.03762v7.pdf', 'page': 1, 'type': 'text', 'bbox': {'l': 124.313, 't': 717.8198352, 'r': 487.89454240000015, 'b': 679.662512679646}}, page_content='Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.')]

In [13]:
# Chunking is working now. Let us look at embeddings next.
# We have already built the embedding code for local, Ollama and OpenAI. We can test them out in the notebook.

In [14]:
text = "This is a sample chunk of text to test embeddings."

In [22]:
embedding_from_local = LocalEmbedder().embed(text)
print("Embedding from Local Embedder:", embedding_from_local)
print("Length of embedding from Local Embedder:", len(embedding_from_local))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11087.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding from Local Embedder: [-0.03855077177286148, -0.022617153823375702, 0.02060004509985447, 0.002727205166593194, 0.07183349132537842, 0.025816230103373528, -0.049323517829179764, -0.027199596166610718, 0.015126246958971024, -0.029482513666152954, 0.04081033915281296, 0.03890411928296089, 0.03307720273733139, 0.018899811431765556, -0.05602564290165901, 0.02303394116461277, 0.1097772940993309, -0.006708569824695587, -0.030583586543798447, -0.003281248267740011, 0.009427238255739212, 0.05365578085184097, 0.0660911574959755, -0.062336817383766174, 0.06178012490272522, 0.023983128368854523, -0.09417262673377991, 0.026000693440437317, 0.0772445797920227, -0.026859307661652565, 0.05570315569639206, -0.006879075430333614, 0.04576549306511879, 0.09590707719326019, 0.060295943170785904, 0.04666213318705559, -0.0037810595240443945, 0.01846526376903057, -0.008489061146974564, 0.06879576295614243, 0.014429685659706593, -0.025337349623441696, 0.019277317449450493, 0.059236809611320496, 0.0318

In [23]:
embedding_from_ollama = OllamaEmbedder(model="nomic-embed-text-v2-moe:latest").embed(text)
print("Embedding from Ollama Embedder:", embedding_from_ollama)
print("Length of embedding from Ollama Embedder:", len(embedding_from_ollama))

Embedding from Ollama Embedder: [[0.4624090790748596, 0.438685804605484, -0.6283262968063354, -0.13556265830993652, 0.4811842143535614, 0.2209518551826477, -0.5395461320877075, -0.3284450173377991, -0.13200829923152924, 1.0935373306274414, -0.3569122850894928, -0.08942488580942154, -0.19411176443099976, 1.1270530223846436, -0.20968122780323029, 0.1023792177438736, 0.33554983139038086, 1.132880687713623, 0.21557188034057617, 0.42443639039993286, -0.5275087356567383, 0.6947552561759949, -0.5911840796470642, -1.0324621200561523, -0.15071195363998413, -0.36380842328071594, -0.03345923125743866, -0.005859009921550751, 0.4706856310367584, -0.023437799885869026, 0.038354381918907166, -0.3518972098827362, 0.16991330683231354, -0.09238128364086151, -0.24249573051929474, -0.0017112195491790771, 0.5081285238265991, 0.07023615390062332, 0.8710798025131226, -0.025319550186395645, 0.534842848777771, -0.48017027974128723, -0.28710630536079407, 0.2067754715681076, -0.5243992805480957, 0.34691482782363

In [26]:
embedding_from_openai = OpenAIEmbedder(model="text-embedding-3-small").embed(text)
print("Embedding from OpenAI Embedder:", embedding_from_openai)

Embedding from OpenAI Embedder: [[0.003143310546875, -0.01136016845703125, 0.01910400390625, 0.015899658203125, -0.02789306640625, -0.046600341796875, 0.0279693603515625, 0.01399993896484375, 0.00785064697265625, 0.019073486328125, 0.04864501953125, -0.0082855224609375, -0.04052734375, -0.036865234375, 0.003894805908203125, 0.033447265625, -0.0175933837890625, 0.01242828369140625, -0.0236968994140625, 0.00958251953125, -0.005916595458984375, -0.0222625732421875, 0.0009737014770507812, 0.044708251953125, 0.003902435302734375, -0.0270843505859375, 0.024688720703125, 0.0212860107421875, 0.052001953125, -0.036773681640625, -0.0164031982421875, -0.04290771484375, 0.00763702392578125, -0.043853759765625, -0.0179443359375, 0.034637451171875, -0.0232391357421875, 0.0240325927734375, -0.026275634765625, -0.0119171142578125, -0.002109527587890625, -0.0178985595703125, -0.00963592529296875, 0.035888671875, -0.0291748046875, 0.03021240234375, -0.046295166015625, -0.0132293701171875, -0.00498199462

In [27]:
print("Length of embedding from OpenAI Embedder:", len(embedding_from_openai[0]))

Length of embedding from OpenAI Embedder: 1536
